In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

def plot_ablation_results():
    # 1. Recriando seus resultados para o plot
    data = {
        'Model': ['Full_Model', 'No_TDA', 'No_Static_Context', 'No_Connectivity', 'No_Climate', 'No_Spatial_Identity'],
        'MAE_R0': [0.211249, 0.197999, 0.189904, 0.189165, 0.192735, 0.187825]
    }
    df = pd.DataFrame(data).sort_values('MAE_R0', ascending=False)

    # 2. Configuração de estilo de artigo científico (clean)
    sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)
    plt.figure(figsize=(10, 6))

    # 3. Criando o gráfico de barras horizontal
    ax = sns.barplot(x="MAE_R0", y="Model", data=df, color="lightgray", edgecolor=".2")

    # 4. Destacando o Pior e o Melhor
    # O Full_Model (Pior) ficará em vermelho escuro. O Campeão em verde escuro.
    for i, model_name in enumerate(df['Model']):
        if model_name == 'Full_Model':
            ax.patches[i].set_facecolor('#d62728') # Vermelho
        elif model_name == 'No_Spatial_Identity':
            ax.patches[i].set_facecolor('#2ca02c') # Verde

    # 5. Adicionando os valores nas barras
    for p in ax.patches:
        ax.annotate(f"{p.get_width():.4f}",
                    (p.get_width() + 0.002, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center', fontsize=11, color='black')

    # 6. Estética final
    plt.title("Estudo de Ablação: Erro Absoluto Médio (MAE) do $R_0$", fontsize=14, fontweight='bold', pad=15)
    plt.xlabel("MAE (Quanto menor, melhor)", fontsize=12)
    plt.ylabel("Configuração do Modelo", fontsize=12)

    # Ajustar limites do eixo X para dar zoom na diferença
    plt.xlim(0.18, 0.22)

    sns.despine(left=True, bottom=True)
    plt.tight_layout()
    plt.savefig("figura_ablation.png", dpi=300, bbox_inches='tight')
    plt.show()

plot_ablation_results()

In [2]:
# =================================================================
# 1. IMPORTS E TRAVA DE SEGURANÇA (PATCH UNIFICADO)
# =================================================================
import os
import glob
import warnings
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer

# Evita RecursionError se a célula for executada múltiplas vezes
if not hasattr(torch, "_is_patched_for_tft"):
    torch._original_load_for_tft = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        ckpt = torch._original_load_for_tft(*args, **kwargs)
        if isinstance(ckpt, dict) and "hyper_parameters" in ckpt:
            # Remove o erro de digitação interno da biblioteca
            ckpt["hyper_parameters"].pop("monotone_constaints", None)
        return ckpt
    torch.load = patched_load
    torch._is_patched_for_tft = True

warnings.filterwarnings("ignore")
torch.set_float32_matmul_precision('medium')

# =================================================================
# 2. CONFIGURAÇÕES E CAPITAIS
# =================================================================
DATA_PATH = "../data/processed/dataset_tft_completo.parquet"
BASE_SAVE_DIR = "results/plots_artigo"

CAPITAIS = {
    "1200401": "Rio_Branco", "2704302": "Maceio", "1600303": "Macapa", "1302603": "Manaus",
    "2927408": "Salvador", "2304400": "Fortaleza", "5300108": "Brasilia", "3205309": "Vitoria",
    "5208707": "Goiania", "2111300": "Sao_Luis", "5103403": "Cuiaba", "5002704": "Campo_Grande",
    "3106200": "Belo_Horizonte", "1501402": "Belem", "2507507": "Joao_Pessoa", "4106902": "Curitiba",
    "2611606": "Recife", "2211001": "Teresina", "3304557": "Rio_de_Janeiro", "2408102": "Natal",
    "4314902": "Porto_Alegre", "1100205": "Porto_Velho", "1400100": "Boa_Vista", "4205407": "Florianopolis",
    "3550308": "Sao_Paulo", "2800308": "Aracaju", "1721000": "Palmas"
}

In [3]:

# =================================================================
# 3. CARREGAMENTO DOS DADOS
# =================================================================
print("⏳ Carregando dados...")
df = pd.read_parquet(DATA_PATH)
df["time_idx"] = df["time_idx"].astype(int)
df["geocode"] = df["geocode"].astype(str)

# Garante tipos categóricos para evitar erros no DataLoader
for col in ["uf", "koppen", "biome", "macroregion_name"]:
    if col in df.columns:
        df[col] = df[col].astype(str)


⏳ Carregando dados...


In [4]:
df_filtered

NameError: name 'df_filtered' is not defined

In [ ]:
# ==========================================
# 4. LOOP DE GERAÇÃO POR MODELO (CORRIGIDO)
# ==========================================
exp_folders = [f for f in os.listdir("models/checkpoints/") if os.path.isdir(os.path.join("models/checkpoints/", f))]

for exp in exp_folders:
    ckpt_path = glob.glob(f"models/checkpoints/{exp}/*.ckpt")
    if not ckpt_path: continue

    print(f"\n📊 Processando: {exp}")
    model = TemporalFusionTransformer.load_from_checkpoint(ckpt_path[0])
    model.eval()

    # --- INÍCIO DA CORREÇÃO ---
    # 1. Identificar quais geocodes o modelo conhece
    known_geocodes = list(model.dataset_parameters['categorical_encoders']['geocode'].classes_.keys())

    # 2. Filtrar o DF para conter apenas as CAPITAIS que o modelo conhece
    # Isso evita o KeyError de categorias desconhecidas
    capitais_codes = list(CAPITAIS.keys())
    codes_to_filter = [c for c in capitais_codes if c in known_geocodes]

    df_filtered = df[df['geocode'].isin(codes_to_filter)].copy()

    print(f"Geocodes filtrados: {df_filtered['geocode'].unique()}")

    if df_filtered.empty:
        print(f"⚠️ Nenhuma capital encontrada no modelo {exp}. Pulando...")
        continue
    # --- FIM DA CORREÇÃO ---

    model_dir = os.path.join(BASE_SAVE_DIR, exp)
    os.makedirs(os.path.join(model_dir, "completo"), exist_ok=True)
    os.makedirs(os.path.join(model_dir, "nao_visto"), exist_ok=True)

    # Preparar Datasets usando o DF FILTRADO
    ds_full = TimeSeriesDataSet.from_parameters(model.dataset_parameters, df_filtered, predict=False)
    dl_full = ds_full.to_dataloader(train=False, batch_size=128, num_workers=0)

    ds_unseen = TimeSeriesDataSet.from_parameters(model.dataset_parameters, df_filtered, predict=True)
    dl_unseen = ds_unseen.to_dataloader(train=False, batch_size=128, num_workers=0)

    # Obter saídas brutas (raw) para plotagem de quantis
    with torch.no_grad():
        out_full = model.predict(dl_full, mode="raw", return_x=True)
        out_unseen = model.predict(dl_unseen, mode="raw", return_x=True)

    geocodes_full = out_full.x["groups"][:, 0].numpy().astype(str)
    geocodes_unseen = out_unseen.x["groups"][:, 0].numpy().astype(str)

    # Plotagem individual para cada capital
    for geocode, nome in tqdm(CAPITAIS.items(), desc=f"Gerando imagens ({exp})"):

        # 1. Gráfico Série Completa
        if geocode in geocodes_full:
            idx = np.where(geocodes_full == geocode)[0][0]
            fig, ax = plt.subplots(figsize=(12, 6))
            model.plot_prediction(out_full.x, out_full.output, idx=idx, ax=ax, add_loss_to_title=False)
            plt.title(f"{nome} ({geocode}) - Histórico e Previsão - Modelo: {exp}")
            plt.savefig(os.path.join(model_dir, "completo", f"{nome}_full.png"), dpi=150)
            plt.close(fig)

        # 2. Gráfico Apenas Não Visto
        if geocode in geocodes_unseen:
            idx = np.where(geocodes_unseen == geocode)[0][0]
            fig, ax = plt.subplots(figsize=(10, 5))
            model.plot_prediction(out_unseen.x, out_unseen.output, idx=idx, ax=ax, add_loss_to_title=False)
            plt.title(f"{nome} ({geocode}) - Previsão Out-of-Sample - Modelo: {exp}")
            plt.savefig(os.path.join(model_dir, "nao_visto", f"{nome}_unseen.png"), dpi=150)
            plt.close(fig)

print(f"\n✅ Concluído! Gráficos salvos em: {BASE_SAVE_DIR}")


📊 Processando: 1_Full_Model
Geocodes filtrados: ['1100205' '1200401' '1302603' '1400100' '1501402' '1600303' '1721000'
 '2111300' '2211001' '2304400' '2408102' '2507507' '2611606' '2704302'
 '2800308' '2927408' '3106200' '3205309' '3304557' '3550308' '4106902'
 '4205407' '4314902' '5002704' '5103403' '5208707' '5300108']


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Gerando imagens (1_Full_Model):   0%|          | 0/27 [00:00<?, ?it/s]


📊 Processando: 2_No_TDA
Geocodes filtrados: ['1100205' '1200401' '1302603' '1400100' '1501402' '1600303' '1721000'
 '2111300' '2211001' '2304400' '2408102' '2507507' '2611606' '2704302'
 '2800308' '2927408' '3106200' '3205309' '3304557' '3550308' '4106902'
 '4205407' '4314902' '5002704' '5103403' '5208707' '5300108']


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

Detected KeyboardInterrupt, attempting graceful shutdown ...


SystemExit: 1

In [1]:
# =================================================================
# 1. IMPORTS E PATCH (UNIFICADO)
# =================================================================
import os, glob, warnings
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer

if not hasattr(torch, "_is_patched_for_tft"):
    torch._original_load_for_tft = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        ckpt = torch._original_load_for_tft(*args, **kwargs)
        if isinstance(ckpt, dict) and "hyper_parameters" in ckpt:
            ckpt["hyper_parameters"].pop("monotone_constaints", None)
        return ckpt
    torch.load = patched_load
    torch._is_patched_for_tft = True

warnings.filterwarnings("ignore")
torch.set_float32_matmul_precision('medium')

# =================================================================
# 2. CONFIGURAÇÕES
# =================================================================
DATA_PATH = "../data/processed/dataset_tft_completo.parquet"
BASE_SAVE_DIR = "results/plots_artigo"

CAPITAIS = {
    "1200401": "Rio_Branco", "2704302": "Maceio", "1600303": "Macapa", "1302603": "Manaus",
    "2927408": "Salvador", "2304400": "Fortaleza", "5300108": "Brasilia", "3205309": "Vitoria",
    "5208707": "Goiania", "2111300": "Sao_Luis", "5103403": "Cuiaba", "5002704": "Campo_Grande",
    "3106200": "Belo_Horizonte", "1501402": "Belem", "2507507": "Joao_Pessoa", "4106902": "Curitiba",
    "2611606": "Recife", "2211001": "Teresina", "3304557": "Rio_de_Janeiro", "2408102": "Natal",
    "4314902": "Porto_Alegre", "1100205": "Porto_Velho", "1400100": "Boa_Vista", "4205407": "Florianopolis",
    "3550308": "Sao_Paulo", "2800308": "Aracaju", "1721000": "Palmas"
}

# =================================================================
# 3. CARREGAMENTO E FILTRAGEM
# =================================================================
df = pd.read_parquet(DATA_PATH)
df["time_idx"] = df["time_idx"].astype(int)
df["geocode"] = df["geocode"].astype(str)
for col in ["uf", "koppen", "biome", "macroregion_name"]:
    if col in df.columns: df[col] = df[col].astype(str)

exp_folders = [f for f in os.listdir("models/checkpoints/") if os.path.isdir(os.path.join("models/checkpoints/", f))]

for exp in exp_folders:
    ckpt_path = glob.glob(f"models/checkpoints/{exp}/*.ckpt")
    if not ckpt_path: continue

    print(f"\n🚀 Processando Modelo: {exp}")
    model = TemporalFusionTransformer.load_from_checkpoint(ckpt_path[0])
    model.eval()

    # Mapeamento do Encoder de Geocode
    geo_encoder = model.dataset_parameters['categorical_encoders']['geocode']
    known_geocodes = list(geo_encoder.classes_.keys())

    # Filtra capitais presentes no modelo
    capitais_presentes = [c for c in CAPITAIS.keys() if c in known_geocodes]
    print(f"✅ Encontradas {len(capitais_presentes)} de {len(CAPITAIS)} capitais neste modelo.")

    df_filtered = df[df['geocode'].isin(capitais_presentes)].copy()

    # Pastas
    model_dir = os.path.join(BASE_SAVE_DIR, exp)
    os.makedirs(os.path.join(model_dir, "completo"), exist_ok=True)
    os.makedirs(os.path.join(model_dir, "nao_visto"), exist_ok=True)

    # Gerar Datasets
    ds_full = TimeSeriesDataSet.from_parameters(model.dataset_parameters, df_filtered, predict=False)
    dl_full = ds_full.to_dataloader(train=False, batch_size=64, num_workers=0)

    ds_unseen = TimeSeriesDataSet.from_parameters(model.dataset_parameters, df_filtered, predict=True)
    dl_unseen = ds_unseen.to_dataloader(train=False, batch_size=64, num_workers=0)

    # Predição e Mapeamento de IDs Reais
    with torch.no_grad():
        out_f = model.predict(dl_full, mode="raw", return_x=True)
        # O SEGREDO: Transformar os índices internos de volta para strings de geocode
        ids_full = geo_encoder.inverse_transform(out_f.x["groups"][:, 0]).astype(str)

        out_u = model.predict(dl_unseen, mode="raw", return_x=True)
        ids_unseen = geo_encoder.inverse_transform(out_u.x["groups"][:, 0]).astype(str)

    # Plotagem
    for geocode, nome in tqdm(CAPITAIS.items(), desc=f"Salvando {exp}"):
        # Plot Série Completa
        if geocode in ids_full:
            idx = np.where(ids_full == geocode)[0][0]
            fig, ax = plt.subplots(figsize=(12, 6))
            model.plot_prediction(out_f.x, out_f.output, idx=idx, ax=ax, add_loss_to_title=False)
            plt.title(f"{nome} - Série Completa - {exp}")
            plt.savefig(os.path.join(model_dir, "completo", f"{nome}_full.png"), dpi=100)
            plt.close(fig)

        # Plot Não Visto
        if geocode in ids_unseen:
            idx = np.where(ids_unseen == geocode)[0][0]
            fig, ax = plt.subplots(figsize=(10, 5))
            model.plot_prediction(out_u.x, out_u.output, idx=idx, ax=ax, add_loss_to_title=False)
            plt.title(f"{nome} - Previsão Out-of-Sample - {exp}")
            plt.savefig(os.path.join(model_dir, "nao_visto", f"{nome}_unseen.png"), dpi=100)
            plt.close(fig)

print(f"\n🏁 Processo Finalizado. Verifique a pasta: {BASE_SAVE_DIR}")


🚀 Processando Modelo: 1_Full_Model
✅ Encontradas 27 de 27 capitais neste modelo.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Salvando 1_Full_Model:   0%|          | 0/27 [00:00<?, ?it/s]


🚀 Processando Modelo: 2_No_TDA
✅ Encontradas 27 de 27 capitais neste modelo.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


...

Salvando 5_No_Connectivity:   0%|          | 0/27 [00:00<?, ?it/s]


🚀 Processando Modelo: 6_No_Static_Context
✅ Encontradas 27 de 27 capitais neste modelo.


💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Salvando 6_No_Static_Context:   0%|          | 0/27 [00:00<?, ?it/s]


🏁 Processo Finalizado. Verifique a pasta: results/plots_artigo


In [6]:
import os, glob, warnings
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer

# =================================================================
# 1. PATCHES E MATEMÁTICA (INCIDÊNCIA)
# =================================================================
if not hasattr(torch, "_is_patched_for_tft"):
    torch._original_load_for_tft = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        ckpt = torch._original_load_for_tft(*args, **kwargs)
        if isinstance(ckpt, dict) and "hyper_parameters" in ckpt:
            ckpt["hyper_parameters"].pop("monotone_constaints", None)
        return ckpt
    torch.load = patched_load
    torch._is_patched_for_tft = True

def richards_incidence(t, L, alpha, beta, peak_idx):
    """Derivada de Richards: Gera o formato de 'sino' para incidência semanal."""
    exp_term = np.exp(-alpha * (t - peak_idx))
    numerator = L * alpha * exp_term
    denominator = np.power(1 + beta * exp_term, (1/beta + 1))
    return numerator / denominator

# =================================================================
# 2. CONFIGURAÇÕES
# =================================================================
DATA_PATH = "../data/processed/dataset_tft_completo.parquet"
BASE_RESULTS_DIR = "results/richards_seasonal_all_models"
ANOS_INTERESSE = [2024, 2025]
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)

# =================================================================
# 3. PROCESSAMENTO DOS DADOS
# =================================================================
df = pd.read_parquet(DATA_PATH)
df["geocode"] = df["geocode"].astype(str)
df["time_idx"] = df["time_idx"].astype(int)

# Com base na imagem image_f802e9.png, a epiweek é YYYYWW
# Precisamos extrair a semana pura para localizar a 42
df['week_number'] = df['epiweek'].astype(str).str[-2:].astype(int)
df['year_number'] = df['epiweek'].astype(str).str[:4].astype(int)

model_folders = [f for f in os.listdir("models/checkpoints/") if os.path.isdir(os.path.join("models/checkpoints/", f))]

# =================================================================
# 4. LOOP MULTI-MODELO
# =================================================================
for model_name in model_folders:
    ckpt_list = glob.glob(f"models/checkpoints/{model_name}/*.ckpt")
    if not ckpt_list: continue

    print(f"\n📂 Processando Modelo: {model_name}")
    model = TemporalFusionTransformer.load_from_checkpoint(ckpt_list[0])
    model.eval()
    geo_encoder = model.dataset_parameters['categorical_encoders']['geocode']

    for ano_foco in ANOS_INTERESSE:
        current_save_path = os.path.join(BASE_RESULTS_DIR, model_name, str(ano_foco))
        os.makedirs(current_save_path, exist_ok=True)

        # LOCALIZAR T0: Semana 42 do ano anterior (Ex: 202342 para a temporada 2024)
        target_epiweek = int(f"{ano_foco - 1}42")
        mask_start = df['epiweek'].astype(int) == target_epiweek

        if mask_start.any():
            start_idx = df[mask_start]['time_idx'].min()
        else:
            # Fallback seguro caso a semana 42 específica falte
            start_idx = df[df['year_number'] == ano_foco]['time_idx'].min() - 10

        for geocode, nome in tqdm(CAPITAIS.items(), desc=f"{model_name} | {ano_foco}", leave=False):
            if geocode not in geo_encoder.classes_: continue

            # Dados da temporada: de Outubro (S42) até o fim do ano de foco
            cap_data = df[(df['geocode'] == geocode) &
                          (df['time_idx'] >= start_idx) &
                          (df['year_number'] <= ano_foco)].sort_values('time_idx')

            if cap_data.empty: continue

            try:
                test_ds = TimeSeriesDataSet.from_parameters(model.dataset_parameters, cap_data, predict=True)
                test_dl = test_ds.to_dataloader(train=False, batch_size=1)

                with torch.no_grad():
                    preds = model.predict(test_dl, mode="prediction")

                # p_peak_week é relativo ao start_idx (Semana 42)
                p_peak_week  = preds[1].item()
                p_L          = np.exp(preds[2].item())
                p_alpha      = preds[3].item()
                p_beta       = preds[4].item()

                # Alinhamento Global
                p_peak_idx_global = start_idx + p_peak_week

                t_range = np.linspace(cap_data['time_idx'].min(), cap_data['time_idx'].max(), 300)
                y_richards = richards_incidence(t_range, p_L, p_alpha, p_beta, p_peak_idx_global)

                # Plotagem
                plt.figure(figsize=(12, 6))
                sns.set_style("white")
                plt.bar(cap_data['time_idx'], cap_data['casos'], alpha=0.3, color='royalblue', label='Casos Reais')
                plt.plot(t_range, y_richards, color='crimson', lw=3, label='Predição Richards (Incidência)')
                plt.axvline(p_peak_idx_global, color='black', linestyle=':', alpha=0.5, label='Pico Previsto')

                plt.title(f"{nome} - Sazonalidade {ano_foco} | Modelo: {model_name}", fontsize=12)
                plt.xlabel("Tempo (time_idx)")
                plt.ylabel("Casos Semanais")
                plt.legend(loc='upper right')

                plt.savefig(os.path.join(current_save_path, f"{nome}_richards_incidence.png"), dpi=120)
                plt.close()
            except Exception: continue

print(f"\n✅ Gráficos sazonais gerados com sucesso em: {BASE_RESULTS_DIR}")


📂 Processando Modelo: 1_Full_Model


1_Full_Model | 2024:   0%|          | 0/27 [00:00<?, ?it/s]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless

1_Full_Model | 2025:   0%|          | 0/27 [00:00<?, ?it/s]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless

...

6_No_Static_Context | 2024:   0%|          | 0/27 [00:00<?, ?it/s]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless

6_No_Static_Context | 2025:   0%|          | 0/27 [00:00<?, ?it/s]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless


✅ Gráficos sazonais gerados com sucesso em: results/richards_seasonal_all_models


In [7]:
import os, glob, warnings
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer

# =================================================================
# 1. MATEMÁTICA E PATCHES
# =================================================================
if not hasattr(torch, "_is_patched_for_tft"):
    torch._original_load_for_tft = torch.load
    def patched_load(*args, **kwargs):
        kwargs['weights_only'] = False
        ckpt = torch._original_load_for_tft(*args, **kwargs)
        if isinstance(ckpt, dict) and "hyper_parameters" in ckpt:
            ckpt["hyper_parameters"].pop("monotone_constaints", None)
        return ckpt
    torch.load = patched_load
    torch._is_patched_for_tft = True

def richards_incidence(t, L, alpha, beta, peak_idx):
    """Derivada de Richards para representar a incidência semanal."""
    exp_term = np.exp(-alpha * (t - peak_idx))
    numerator = L * alpha * exp_term
    denominator = np.power(1 + beta * exp_term, (1/beta + 1))
    return numerator / denominator

# =================================================================
# 2. CONFIGURAÇÕES
# =================================================================
DATA_PATH = "../data/processed/dataset_tft_completo.parquet"
BASE_RESULTS_DIR = "results/previsao_temporada_2025"
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)

CAPITAIS = {
    "1200401": "Rio_Branco", "2704302": "Maceio", "1600303": "Macapa", "1302603": "Manaus",
    "2927408": "Salvador", "2304400": "Fortaleza", "5300108": "Brasilia", "3205309": "Vitoria",
    "5208707": "Goiania", "2111300": "Sao_Luis", "5103403": "Cuiaba", "5002704": "Campo_Grande",
    "3106200": "Belo_Horizonte", "1501402": "Belem", "2507507": "Joao_Pessoa", "4106902": "Curitiba",
    "2611606": "Recife", "2211001": "Teresina", "3304557": "Rio_de_Janeiro", "2408102": "Natal",
    "4314902": "Porto_Alegre", "1100205": "Porto_Velho", "1400100": "Boa_Vista", "4205407": "Florianopolis",
    "3550308": "Sao_Paulo", "2800308": "Aracaju", "1721000": "Palmas"
}

# =================================================================
# 3. CARREGAMENTO DOS DADOS DISPONÍVEIS
# =================================================================
df = pd.read_parquet(DATA_PATH)
df["geocode"] = df["geocode"].astype(str)
df["time_idx"] = df["time_idx"].astype(int)
df['epiweek_str'] = df['epiweek'].astype(str)

# Encontrar o time_idx da Semana 42 de 2024 (Início da Temporada 2025)
# Se não existir no seu DF, calculamos com base no último registro
target_epiweek = "202442"
mask_start = df['epiweek_str'] == target_epiweek

if mask_start.any():
    start_idx_2025 = df[mask_start]['time_idx'].min()
else:
    # Se os dados acabam antes, usamos o último time_idx disponível como referência
    start_idx_2025 = df['time_idx'].max()
    print(f"⚠️ Semana 202442 não encontrada. Usando time_idx {start_idx_2025} como início.")

# =================================================================
# 4. LOOP MULTI-MODELO PARA PREVISÃO
# =================================================================
model_folders = [f for f in os.listdir("models/checkpoints/") if os.path.isdir(os.path.join("models/checkpoints/", f))]

for model_name in model_folders:
    ckpt_list = glob.glob(f"models/checkpoints/{model_name}/*.ckpt")
    if not ckpt_list: continue

    print(f"\n🔮 Gerando Previsões 2025 - Modelo: {model_name}")
    model = TemporalFusionTransformer.load_from_checkpoint(ckpt_list[0])
    model.eval()

    current_save_path = os.path.join(BASE_RESULTS_DIR, model_name)
    os.makedirs(current_save_path, exist_ok=True)

    for geocode, nome in tqdm(CAPITAIS.items(), desc=model_name):
        # Pegamos o histórico mais recente disponível para alimentar o Encoder
        cap_history = df[df['geocode'] == geocode].sort_values('time_idx').tail(52)

        if cap_history.empty: continue

        try:
            # Usamos o histórico para prever os parâmetros da próxima temporada
            test_ds = TimeSeriesDataSet.from_parameters(model.dataset_parameters, cap_history, predict=True)
            test_dl = test_ds.to_dataloader(train=False, batch_size=1)

            with torch.no_grad():
                preds = model.predict(test_dl, mode="prediction")

            # Parâmetros previstos (Multi-Target)
            p_peak_week  = preds[1].item()
            p_L          = np.exp(preds[2].item())
            p_alpha      = preds[3].item()
            p_beta       = preds[4].item()

            # Ponto de pico absoluto no futuro
            p_peak_idx_global = start_idx_2025 + p_peak_week

            # Geração do Eixo X Sintético (52 semanas a partir de 202442)
            t_future = np.linspace(start_idx_2025, start_idx_2025 + 52, 200)
            y_richards = richards_incidence(t_future, p_L, p_alpha, p_beta, p_peak_idx_global)

            # PLOTAGEM
            plt.figure(figsize=(12, 6))
            plt.plot(t_future, y_richards, color='crimson', lw=3, label=f'Previsão Sazonal 2025 ({model_name})')
            plt.axvline(p_peak_idx_global, color='black', linestyle='--', alpha=0.4, label='Pico Estimado')

            # Plotar o finalzinho do histórico real (se disponível) para contexto
            plt.plot(cap_history['time_idx'].tail(5), cap_history['casos'].tail(5), color='royalblue', alpha=0.5, label='Fim do Histórico')

            plt.title(f"Previsão Semanal 2025: {nome} (Semana 42/2024 - 42/2025)", fontsize=13)
            plt.xlabel("Tempo (time_idx)")
            plt.ylabel("Casos Semanais Estimados")
            plt.legend()
            plt.grid(alpha=0.2)

            plt.savefig(os.path.join(current_save_path, f"{nome}_pred_2025.png"), dpi=120)
            plt.close()

        except Exception:
            continue

print(f"\n✅ Previsões para 2025 geradas em: {BASE_RESULTS_DIR}")


🔮 Gerando Previsões 2025 - Modelo: 1_Full_Model


1_Full_Model:   0%|          | 0/27 [00:00<?, ?it/s]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless


🔮 Gerando Previsões 2025 - Modelo: 2_No_TDA


2_No_TDA:   0%|          | 0/27 [00:00<?, ?it/s]

...

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless


🔮 Gerando Previsões 2025 - Modelo: 6_No_Static_Context


6_No_Static_Context:   0%|          | 0/27 [00:00<?, ?it/s]

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
💡 Tip: For seamless


✅ Previsões para 2025 geradas em: results/previsao_temporada_2025
